In [2]:
import os
import scipy.io as sio
from scipy.io.matlab._mio5_params import mat_struct
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split

from pathlib import Path
import re
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

In [3]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [4]:
!ls "/content/drive/My Drive/ASCE_IASC"

figures					random_window_split_seed42.csv
MAT_data				Results
processed_zero_mean			txt_all_15_sensors
random_block_assignment_45s_seed42.csv	txt_selected_sensors


In [5]:
from pathlib import Path

import numpy as np
import scipy.io as sio


# مسیر فایل‌ها
DATA_DIR = Path("/content/drive/MyDrive/ASCE_IASC/MAT_data")

# فقط 15 سنسور واقعی سازه
CHANNELS = [f"DA{i:02d}" for i in range(1, 16)]

# پیدا کردن فایل‌های 9 حالت سازه
mat_files = sorted(DATA_DIR.glob("shm*.mat"))

print("Folder exists:", DATA_DIR.exists())
print("Number of files:", len(mat_files))
print("Number of channels:", len(CHANNELS))


def load_ambient_file(file_path):
    """
    خواندن یک فایل Ambient از داده‌های IASC-ASCE
    """

    mat = sio.loadmat(file_path,squeeze_me=True,struct_as_record=False)

    dasy = mat["dasy"]
    dasy_dscr = mat["dasy_dscr"]
    fs = float(mat["fsdasy"])

    signals = []
    descriptions = []

    for i, channel in enumerate(CHANNELS, start=1):

        signal = np.asarray(getattr(dasy, channel)).reshape(-1)

        description_name = f"DAdscr{i:02d}"
        description = getattr(dasy_dscr, description_name)

        signals.append(signal)
        descriptions.append(description)

    # شکل خروجی:
    # (تعداد نمونه‌های زمانی، تعداد سنسورها)
    X_raw = np.column_stack(signals)

    return X_raw, fs, descriptions

Folder exists: True
Number of files: 9
Number of channels: 15


In [6]:
all_cases = {}

for file_path in mat_files:

    case_name = file_path.stem
    case_number = int(case_name[3:5])

    X_raw, fs, descriptions = load_ambient_file(file_path)

    all_cases[case_name] = {
        "X_raw": X_raw,
        "fs": fs,
        "channels": CHANNELS.copy(),
        "descriptions": descriptions,
        "case_number": case_number,
        "label": case_number - 1,
        "file_path": file_path
    }

    print(
        f"{case_name} | "
        f"shape: {X_raw.shape} | "
        f"fs: {fs} Hz | "
        f"label: {case_number - 1}"

    )
print( f"description: {pd.DataFrame(descriptions)}")

shm01a | shape: (60000, 15) | fs: 200.0 Hz | label: 0
shm02a | shape: (60000, 15) | fs: 200.0 Hz | label: 1
shm03a | shape: (60000, 15) | fs: 200.0 Hz | label: 2
shm04a | shape: (60000, 15) | fs: 200.0 Hz | label: 3
shm05a | shape: (60000, 15) | fs: 200.0 Hz | label: 4
shm06a | shape: (45568, 15) | fs: 200.0 Hz | label: 5
shm07a | shape: (180000, 15) | fs: 200.0 Hz | label: 6
shm08a | shape: (180000, 15) | fs: 200.0 Hz | label: 7
shm09a | shape: (180000, 15) | fs: 200.0 Hz | label: 8
description:                                                0
0   Base West side - EPI sensor X direction (N+)
1      Base Center - EPI sensor Y direction (W+)
2   Base East side - EPI sensor X direction (N+)
3    1st Floor - N/S EPI Sensor at West end (N+)
4      1st Floor - E/W FBA Sensor at Center (W+)
5    1st Floor - N/S FBA Sensor at East end (N+)
6    2nd Floor - N/S FBA Sensor at West end (N+)
7      2nd Floor - E/W EPI Sensor at Center (W+)
8    2nd Floor - N/S EPI Sensor at East end (N+)
9    3rd

In [7]:
print(all_cases["shm01a"]["X_raw"].shape)
print(all_cases["shm01a"]["channels"])


(60000, 15)
['DA01', 'DA02', 'DA03', 'DA04', 'DA05', 'DA06', 'DA07', 'DA08', 'DA09', 'DA10', 'DA11', 'DA12', 'DA13', 'DA14', 'DA15']


In [8]:
# ============================================================
#  Step 2: Random Block Split settings
# ============================================================

import pandas as pd
from scipy.signal import butter, sosfiltfilt
import random


# ----------------------------
# Reproducibility
# ----------------------------
SPLIT_SEED = 42
MODEL_SEED = 42

def set_seed(seed):

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    # Reproducibility for CUDA
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    # Warn if a nondeterministic operation is used
    torch.use_deterministic_algorithms(True,warn_only=True)

set_seed(MODEL_SEED)



# ----------------------------
# Block split settings
# ----------------------------
BLOCK_SEC = 45
TRAIN_RATIO = 0.60
VAL_RATIO = 0.20

# GAP_SEC = 0
EDGE_SEC = 2


# ----------------------------
# High-pass filter settings
# ----------------------------
USE_HIGHPASS = True
CUTOFF = 0.1
ORDER = 2


def highpass_filter(X, fs):

    """
    Apply a Butterworth high-pass filter to all sensor channels.
    X shape:(samples, sensors)
    """

    sos = butter(N=ORDER, Wn=CUTOFF,btype="highpass",fs=fs,output="sos")
    X_filtered = sosfiltfilt(sos,X,axis=0)

    return X_filtered

In [9]:
# ============================================================
# Step 4: Create fixed random block assignments
# ============================================================

assignment_rows = []
block_summary_rows = []

for case_name in sorted(all_cases.keys()):

    X_raw = all_cases[case_name]["X_raw"]
    fs = all_cases[case_name]["fs"]

    n_samples = len(X_raw)
    block_size = int(BLOCK_SEC * fs)

    # فقط بلوک‌های کامل
    number_of_blocks = (len(X_raw) // block_size)

    if number_of_blocks < 5:
        raise ValueError(f"{case_name}: fewer than five complete blocks.")

    used_samples = (number_of_blocks * block_size)
    unused_samples = (n_samples - used_samples )

    # ----------------------------
    # Number of blocks per split
    # ----------------------------
    number_of_train_blocks = int(round(TRAIN_RATIO * number_of_blocks))
    number_of_val_blocks = int(round(VAL_RATIO * number_of_blocks))
    number_of_test_blocks = (number_of_blocks- number_of_train_blocks- number_of_val_blocks)

    # حداقل یک بلوک برای هر Split
    if number_of_train_blocks < 1: number_of_train_blocks = 1
    if number_of_val_blocks < 1: number_of_val_blocks = 1

    # ----------------------------
    # Random block order
    # A separate deterministic seed per case
    # ----------------------------
    scenario_number = int(case_name[3:5])
    case_generator = np.random.default_rng(SPLIT_SEED + scenario_number)
    shuffled_ids = (case_generator.permutation(number_of_blocks))

    train_ids = set(shuffled_ids[:number_of_train_blocks])
    val_ids = set(shuffled_ids[number_of_train_blocks:number_of_train_blocks+ number_of_val_blocks])
    test_ids = set(shuffled_ids[number_of_train_blocks+ number_of_val_blocks:])

    # ----------------------------
    # Save block assignments
    # ----------------------------
    for block_id in range(number_of_blocks):

        start_sample = (block_id * block_size)
        end_sample = (start_sample + block_size)

        if block_id in train_ids:
            split_name = "train"

        elif block_id in val_ids:
            split_name = "validation"

        elif block_id in test_ids:
            split_name = "test"

        else:
            raise RuntimeError("Block assignment failed.")

        start_sample = block_id * block_size
        end_sample = start_sample + block_size

        assignment_rows.append({
            "case": case_name,
            "block_id": block_id,
            "start_sample": start_sample,
            "end_sample": end_sample,
            "start_sec": start_sample / fs,
            "end_sec": end_sample / fs,
            "split": split_name
        })
    used_samples = number_of_blocks * block_size
    unused_samples = n_samples - used_samples

    block_summary_rows.append({
        "case": case_name,
        "total_blocks": number_of_blocks,
        "train_blocks": number_of_train_blocks,
        "validation_blocks": number_of_val_blocks,
        "test_blocks": number_of_test_blocks,
        "unused_seconds": unused_samples / fs
    })

In [11]:
block_assignment_df = pd.DataFrame(assignment_rows)
display(block_assignment_df.head(20))

block_summary_df = pd.DataFrame(block_summary_rows)
display(block_summary_df)

,case,block_id,start_sample,end_sample,start_sec,end_sec,split
0,shm01a,0,0,9000,0.0,45.0,validation
1,shm01a,1,9000,18000,45.0,90.0,train
2,shm01a,2,18000,27000,90.0,135.0,train
3,shm01a,3,27000,36000,135.0,180.0,test
4,shm01a,4,36000,45000,180.0,225.0,train
5,shm01a,5,45000,54000,225.0,270.0,train
6,shm02a,0,0,9000,0.0,45.0,train
7,shm02a,1,9000,18000,45.0,90.0,test
8,shm02a,2,18000,27000,90.0,135.0,train
9,shm02a,3,27000,36000,135.0,180.0,train


,case,total_blocks,train_blocks,validation_blocks,test_blocks,unused_seconds
0,shm01a,6,4,1,1,30.00
1,shm02a,6,4,1,1,30.00
2,shm03a,6,4,1,1,30.00
3,shm04a,6,4,1,1,30.00
4,shm05a,6,4,1,1,30.00
5,shm06a,5,3,1,1,2.84
6,shm07a,20,12,4,4,0.00
7,shm08a,20,12,4,4,0.00
8,shm09a,20,12,4,4,0.00


In [12]:
block_count_df = (block_assignment_df.groupby(["case", "split"]).size().unstack(fill_value=0))

display(block_count_df)

split,test,train,validation
case,,,
shm01a,1,4,1
shm02a,1,4,1
shm03a,1,4,1
shm04a,1,4,1
shm05a,1,4,1
shm06a,1,3,1
shm07a,4,12,4
shm08a,4,12,4
shm09a,4,12,4


In [13]:
BLOCK_ASSIGNMENT_PATH = (
    "/content/drive/MyDrive/ASCE_IASC/"
    f"random_block_assignment_45s_seed{SPLIT_SEED}.csv")

block_assignment_df.to_csv(BLOCK_ASSIGNMENT_PATH,index=False)
print("Block assignment saved:",BLOCK_ASSIGNMENT_PATH)

Block assignment saved: /content/drive/MyDrive/ASCE_IASC/random_block_assignment_45s_seed42.csv


In [ ]:
# block_assignment_df = pd.read_csv(BLOCK_ASSIGNMENT_PATH)

In [14]:
# ============================================================
# Build Train, Validation and Test block lists
# ============================================================

train_blocks = []
val_blocks = []
test_blocks = []


for _, row in block_assignment_df.iterrows():

    case_name = row["case"]
    block_id = int(row["block_id"])
    split_name = row["split"]

    start_sample = int(row["start_sample"])
    end_sample = int(row["end_sample"])

    X_raw = all_cases[case_name]["X_raw"]
    fs = all_cases[case_name]["fs"]
    edge_size = int(EDGE_SEC * fs)

    # ----------------------------
    # Extract one raw block
    # ----------------------------
    X_block = X_raw[start_sample:end_sample]

    # ----------------------------
    # Optional high-pass filter
    # ----------------------------
    if USE_HIGHPASS:
        X_block = highpass_filter(X_block,fs)

    else:
        X_block = X_block.copy()

    # ----------------------------
    # Matched edge removal
    # ----------------------------
    X_block = X_block[edge_size:-edge_size]

    block_record = {
        "case": case_name,
        "block_id": block_id,
        "block_start_sample": start_sample,
        "block_end_sample": end_sample,
        "data": X_block
    }

    if split_name == "train":
        train_blocks.append(block_record)

    elif split_name == "validation":
        val_blocks.append(block_record)

    elif split_name == "test":
        test_blocks.append(block_record)

In [15]:
train_blocks[0].keys()

dict_keys(['case', 'block_id', 'block_start_sample', 'block_end_sample', 'data'])

In [16]:
print("Train blocks:", len(train_blocks))
print("Validation blocks:", len(val_blocks))
print("Test blocks:", len(test_blocks))

Train blocks: 59
Validation blocks: 18
Test blocks: 18


In [17]:
print("Example block shape:",
    train_blocks[0]["data"].shape)
# (7000, 15)

Example block shape: (8200, 15)


In [18]:
# ============================================================
# Step 3: Select sensor channels
# ============================================================

# selected_sensor_indices = [8, 11, 13, 14]  # DA09, DA12, DA14, DA15
# selected_sensor_indices = [0,3]
selected_sensor_indices = [3,4,5,6,7,8,9,10,11,12,13,14]

selected_sensor_names = [CHANNELS[i] for i in selected_sensor_indices]



In [19]:
def select_sensors(block_list):
    """
    Select sensor columns from every block.
    """

    selected_list = []

    for block in block_list:

        selected_block = {
            "case": block["case"],
            "block_id": block["block_id"],
            "block_start_sample": block["block_start_sample"],
            "block_end_sample": block["block_end_sample"],
            "data": block["data"][:, selected_sensor_indices]
        }

        selected_list.append(selected_block)

    return selected_list

In [20]:
train_selected = select_sensors(train_blocks)
val_selected = select_sensors(val_blocks)
test_selected = select_sensors(test_blocks)

In [21]:
print(
    "Selected Train block shape:",
    train_selected[0]["data"].shape
)

Selected Train block shape: (8200, 12)


In [22]:
train_selected[0].keys()

dict_keys(['case', 'block_id', 'block_start_sample', 'block_end_sample', 'data'])

In [23]:
# ============================================================
# Step 7: Global train-based normalization
# One mean and standard deviation for each selected sensor
# ============================================================

EPS = 1e-8

train_concat = np.concatenate(
    [ block["data"] for block in train_selected], axis=0)


print("Combined Train shape:",train_concat.shape)

Combined Train shape: (483800, 12)


In [24]:
train_mean = train_concat.mean(axis=0, keepdims=True, dtype=np.float64)
train_std = train_concat.std( axis=0,keepdims=True,dtype=np.float64)

# Prevent division by zero
train_std_safe = np.where(train_std < EPS,1.0,train_std)

print("Train mean shape:", train_mean.shape)
print("Train std shape:", train_std_safe.shape)

Train mean shape: (1, 12)
Train std shape: (1, 12)


In [29]:
def normalize_blocks(block_list):

    normalized_blocks = []

    for block in block_list:

        X_normalized = ((block["data"]-train_mean)/train_std_safe).astype(np.float32)

        normalized_block = {
          "case": block["case"],
          "block_id": block["block_id"],
          "block_start_sample": block["block_start_sample"],
          "block_end_sample": block["block_end_sample"],
          "data": X_normalized
      }

        normalized_blocks.append(normalized_block)

    return normalized_blocks

In [30]:
train_norm = normalize_blocks(train_selected)
val_norm = normalize_blocks(val_selected)
test_norm = normalize_blocks(test_selected)

print("Normalization completed.")

Normalization completed.


In [31]:
train_norm[0].keys()

dict_keys(['case', 'block_id', 'block_start_sample', 'block_end_sample', 'data'])

In [32]:
train_norm_concat = np.concatenate(
    [ block["data"] for block in train_norm],axis=0)


print("Train mean after normalization:",
    np.round(train_norm_concat.mean(axis=0),5))

print("Train std after normalization:",
    np.round(train_norm_concat.std(axis=0),5))

Train mean after normalization: [ 0.  0.  0.  0.  0.  0.  0.  0. -0.  0. -0. -0.]
Train std after normalization: [1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


In [33]:
# ============================================================
# Step 5: Create non-overlapping time windows
# ============================================================

WINDOW_SIZE = 512
STEP_SIZE = 512

FS_DASY = 200

print("Window size:", WINDOW_SIZE, "samples")
print("Window duration:", WINDOW_SIZE / FS_DASY, "seconds")
print("Step size:", STEP_SIZE, "samples")
print("Step duration:", STEP_SIZE / FS_DASY, "seconds")

Window size: 512 samples
Window duration: 2.56 seconds
Step size: 512 samples
Step duration: 2.56 seconds


In [34]:
def make_windows(split_data, split_name):

    all_windows = []
    all_labels = []
    meta_rows = []

    for block in split_data:

        case_name = block["case"]
        block_id = block["block_id"]

        block_start = block["block_start_sample"]
        block_end = block["block_end_sample"]

        X = block["data"]

        fs = all_cases[case_name]["fs"]
        edge_size = int(EDGE_SEC * fs)

        scenario_number = int(case_name[3:5])
        label = scenario_number - 1

        window_number = 0

        for start in range(0, len(X) - WINDOW_SIZE + 1, STEP_SIZE ):

            end = start + WINDOW_SIZE

            window = X[start:end, :]

            all_windows.append(window)
            all_labels.append(label)

            # Position in original raw recording
            global_start = (block_start+ edge_size+ start)
            global_end = global_start + WINDOW_SIZE

            meta_rows.append({
                "case": case_name,
                "scenario": scenario_number,
                "label": label,
                "block_id": block_id,
                "window": window_number,
                "split": split_name,
                "start_sample_in_block": start,
                "end_sample_in_block": end,
                "global_start_sample": global_start,
                "global_end_sample": global_end
            })

            window_number += 1

    if len(all_windows) == 0:
        raise ValueError("No windows were created.")

    X_windows = np.stack(all_windows,axis=0).astype(np.float32)
    y_labels = np.array(all_labels,dtype=np.int64)
    meta_df = pd.DataFrame(meta_rows)

    return X_windows, y_labels, meta_df

In [35]:
X_train, y_train, train_meta_df = make_windows(train_norm, "train")
X_val, y_val, val_meta_df = make_windows(val_norm, "val")
X_test, y_test, test_meta_df = make_windows(test_norm, "test")

In [36]:
display(train_meta_df.head(3))
display(val_meta_df.head(3))
display(test_meta_df.head(3))

,case,scenario,label,block_id,window,split,start_sample_in_block,end_sample_in_block,global_start_sample,global_end_sample
0,shm01a,1,0,1,0,train,0,512,9400,9912
1,shm01a,1,0,1,1,train,512,1024,9912,10424
2,shm01a,1,0,1,2,train,1024,1536,10424,10936


,case,scenario,label,block_id,window,split,start_sample_in_block,end_sample_in_block,global_start_sample,global_end_sample
0,shm01a,1,0,0,0,val,0,512,400,912
1,shm01a,1,0,0,1,val,512,1024,912,1424
2,shm01a,1,0,0,2,val,1024,1536,1424,1936


,case,scenario,label,block_id,window,split,start_sample_in_block,end_sample_in_block,global_start_sample,global_end_sample
0,shm01a,1,0,3,0,test,0,512,27400,27912
1,shm01a,1,0,3,1,test,512,1024,27912,28424
2,shm01a,1,0,3,2,test,1024,1536,28424,28936


In [38]:
# ============================================================
# Structural Audit 1: Check block assignment integrity
# ============================================================

block_split_check = (
    block_assignment_df
    .groupby(["case", "block_id"])["split"]
    .nunique()
    .reset_index(name="number_of_splits")
)

problem_blocks = block_split_check[
    block_split_check["number_of_splits"] > 1
]

print("Total blocks:", len(block_split_check))
print("Blocks assigned to more than one split:", len(problem_blocks))

display(problem_blocks)

Total blocks: 95
Blocks assigned to more than one split: 0


,case,block_id,number_of_splits


In [39]:
# ============================================================
# Structural Audit 2: Combine window metadata
# ============================================================

audit_meta_df = pd.concat(
    [train_meta_df, val_meta_df, test_meta_df],
    ignore_index=True
)

print("Total windows:", len(audit_meta_df))

display(audit_meta_df.head())

Total windows: 1520


,case,scenario,label,block_id,window,split,start_sample_in_block,end_sample_in_block,global_start_sample,global_end_sample
0,shm01a,1,0,1,0,train,0,512,9400,9912
1,shm01a,1,0,1,1,train,512,1024,9912,10424
2,shm01a,1,0,1,2,train,1024,1536,10424,10936
3,shm01a,1,0,1,3,train,1536,2048,10936,11448
4,shm01a,1,0,1,4,train,2048,2560,11448,11960


In [40]:
# ============================================================
# Structural Audit 3: Basic sanity checks
# ============================================================

assert len(train_meta_df) == len(X_train)
assert len(val_meta_df) == len(X_val)
assert len(test_meta_df) == len(X_test)

window_lengths = (
    audit_meta_df["global_end_sample"]
    - audit_meta_df["global_start_sample"]
)

assert (window_lengths == WINDOW_SIZE).all()

for case_name in sorted(all_cases.keys()):

    n_samples = len(all_cases[case_name]["X_raw"])

    case_meta = audit_meta_df[
        audit_meta_df["case"] == case_name
    ]

    assert (case_meta["global_start_sample"] >= 0).all()
    assert (case_meta["global_end_sample"] <= n_samples).all()

print("All basic structural checks passed.")

All basic structural checks passed.


In [41]:
# ============================================================
# Structural Audit 4: Cross-split raw sample overlap
# ============================================================

split_pairs = [
    ("train", "val"),
    ("train", "test"),
    ("val", "test")
]

overlap_rows = []

for case_name in sorted(all_cases.keys()):

    case_meta = audit_meta_df[
        audit_meta_df["case"] == case_name
    ]

    for split_a, split_b in split_pairs:

        meta_a = case_meta[
            case_meta["split"] == split_a
        ]

        meta_b = case_meta[
            case_meta["split"] == split_b
        ]

        overlap_samples = 0
        windows_a_with_overlap = set()
        windows_b_with_overlap = set()

        for idx_a, row_a in meta_a.iterrows():

            for idx_b, row_b in meta_b.iterrows():

                overlap = max(
                    0,
                    min(
                        row_a["global_end_sample"],
                        row_b["global_end_sample"]
                    )
                    -
                    max(
                        row_a["global_start_sample"],
                        row_b["global_start_sample"]
                    )
                )

                if overlap > 0:

                    overlap_samples += overlap

                    windows_a_with_overlap.add(idx_a)
                    windows_b_with_overlap.add(idx_b)

        overlap_rows.append({
            "case": case_name,
            "split_a": split_a,
            "split_b": split_b,
            "overlap_samples": int(overlap_samples),
            "windows_a_with_overlap": len(windows_a_with_overlap),
            "windows_b_with_overlap": len(windows_b_with_overlap)
        })


cross_split_overlap_df = pd.DataFrame(overlap_rows)

display(cross_split_overlap_df)

,case,split_a,split_b,overlap_samples,windows_a_with_overlap,windows_b_with_overlap
0,shm01a,train,val,0,0,0
1,shm01a,train,test,0,0,0
2,shm01a,val,test,0,0,0
3,shm02a,train,val,0,0,0
4,shm02a,train,test,0,0,0
5,shm02a,val,test,0,0,0
6,shm03a,train,val,0,0,0
7,shm03a,train,test,0,0,0
8,shm03a,val,test,0,0,0
9,shm04a,train,val,0,0,0


In [42]:
# ============================================================
# Cross-split overlap summary
# ============================================================

overlap_summary_df = (
    cross_split_overlap_df
    .groupby(["split_a", "split_b"], as_index=False)
    [
        [
            "overlap_samples",
            "windows_a_with_overlap",
            "windows_b_with_overlap"
        ]
    ]
    .sum()
)

display(overlap_summary_df)

,split_a,split_b,overlap_samples,windows_a_with_overlap,windows_b_with_overlap
0,train,test,0,0,0
1,train,val,0,0,0
2,val,test,0,0,0


In [43]:
# ============================================================
# Structural Audit 5: Window counts by scenario and split
# ============================================================

window_count_df = (
    audit_meta_df
    .groupby(["scenario", "split"])
    .size()
    .unstack(fill_value=0)
)

window_count_df = window_count_df[
    ["train", "val", "test"]
]

display(window_count_df)

print("Total windows per split:")

display(
    audit_meta_df["split"]
    .value_counts()
    .reindex(["train", "val", "test"])
)

split,train,val,test
scenario,,,
1,64,16,16
2,64,16,16
3,64,16,16
4,64,16,16
5,64,16,16
6,48,16,16
7,192,64,64
8,192,64,64
9,192,64,64


Total windows per split:


,count
split,
train,944
val,288
test,288


In [44]:
# ============================================================
# Structural Audit 6: Test-to-Train temporal distance
# Same case only
# ============================================================

temporal_rows = []

for case_name in sorted(all_cases.keys()):

    train_case = train_meta_df[
        train_meta_df["case"] == case_name
    ].reset_index(drop=True)

    test_case = test_meta_df[
        test_meta_df["case"] == case_name
    ].reset_index(drop=True)

    fs = all_cases[case_name]["fs"]

    train_starts = train_case["global_start_sample"].to_numpy()
    train_ends = train_case["global_end_sample"].to_numpy()

    for _, test_row in test_case.iterrows():

        test_start = test_row["global_start_sample"]
        test_end = test_row["global_end_sample"]

        distances = np.maximum(
            0,
            np.maximum(
                train_starts - test_end,
                test_start - train_ends
            )
        )

        nearest_pos = np.argmin(distances)
        min_distance = int(distances[nearest_pos])

        temporal_rows.append({
            "case": case_name,
            "scenario": test_row["scenario"],
            "test_block": test_row["block_id"],
            "test_window": test_row["window"],
            "nearest_train_block":
                train_case.iloc[nearest_pos]["block_id"],
            "nearest_train_window":
                train_case.iloc[nearest_pos]["window"],
            "distance_samples": min_distance,
            "distance_sec": min_distance / fs
        })

temporal_distance_df = pd.DataFrame(temporal_rows)

display(temporal_distance_df.head())

,case,scenario,test_block,test_window,nearest_train_block,nearest_train_window,distance_samples,distance_sec
0,shm01a,1,3,0,2,15,808,4.04
1,shm01a,1,3,1,2,15,1320,6.60
2,shm01a,1,3,2,2,15,1832,9.16
3,shm01a,1,3,3,2,15,2344,11.72
4,shm01a,1,3,4,2,15,2856,14.28


In [45]:
temporal_by_case_df = (
    temporal_distance_df
    .groupby(["case", "scenario"])["distance_sec"]
    .agg(["min", "median", "mean"])
    .reset_index()
)

display(temporal_by_case_df)

,case,scenario,min,median,mean
0,shm01a,1,4.04,13.00,13.00
1,shm02a,2,4.04,13.00,13.00
2,shm03a,3,4.04,23.24,23.24
3,shm04a,4,4.04,23.24,23.24
4,shm05a,5,4.04,13.00,13.00
5,shm06a,6,4.04,23.24,23.24
6,shm07a,7,4.04,16.84,18.12
7,shm08a,8,4.04,68.24,65.68
8,shm09a,9,4.04,19.40,20.68


In [47]:
overall_temporal_summary = pd.DataFrame({
    "minimum_sec": [
        temporal_distance_df["distance_sec"].min()
    ],
    "median_sec": [
        temporal_distance_df["distance_sec"].median()
    ],
    "mean_sec": [
        temporal_distance_df["distance_sec"].mean()
    ]
})

display(overall_temporal_summary)

,minimum_sec,median_sec,mean_sec
0,4.04,19.4,29.257778


In [49]:
# ============================================================
# Structural Audit 7: Test windows adjacent to Train
# ============================================================

adjacent_count = (
    temporal_distance_df["distance_samples"] == 0
).sum()

total_test = len(temporal_distance_df)

print("Total Test windows:", total_test)
print("Adjacent Test windows:", adjacent_count)
print(
    f"Adjacent Test windows (%): "
    f"{100 * adjacent_count / total_test:.2f}"
)

Total Test windows: 288
Adjacent Test windows: 0
Adjacent Test windows (%): 0.00


In [50]:
# ============================================================
# Structural Audit 8: Raw data utilization
# ============================================================

total_raw = sum(
    len(all_cases[case]["X_raw"])
    for case in all_cases
)

total_used = len(audit_meta_df) * WINDOW_SIZE

used_percent = 100 * total_used / total_raw

print("Total raw samples:", total_raw)
print("Samples used in model windows:", total_used)
print(f"Overall raw data used (%): {used_percent:.2f}")

Total raw samples: 885568
Samples used in model windows: 778240
Overall raw data used (%): 87.88


In [51]:
# ============================================================
# Similarity Audit 1A: Create unnormalized audit windows
# High-pass + edge trim + selected sensors
# ============================================================

X_train_audit, y_train_audit, train_audit_meta = make_windows(
    train_selected, "train"
)

X_val_audit, y_val_audit, val_audit_meta = make_windows(
    val_selected, "val"
)

X_test_audit, y_test_audit, test_audit_meta = make_windows(
    test_selected, "test"
)

print("Train:", X_train_audit.shape)
print("Val:  ", X_val_audit.shape)
print("Test: ", X_test_audit.shape)

Train: (944, 512, 12)
Val:   (288, 512, 12)
Test:  (288, 512, 12)


In [52]:
# ============================================================
# Similarity Audit 1B: Statistical features
# Output: Windows x 4 Features x 12 Sensors
# ============================================================

from scipy.stats import kurtosis

feature_names = [
    "RMS",
    "STD",
    "Peak-to-peak",
    "Kurtosis"
]

def extract_stat_features(X_windows):

    n_windows = X_windows.shape[0]
    n_sensors = X_windows.shape[2]

    features = np.zeros((n_windows, 4, n_sensors), dtype=np.float64)

    for i in range(n_windows):

        for j in range(n_sensors):

            x = X_windows[i, :, j]

            features[i, 0, j] = np.sqrt(np.mean(x ** 2))
            features[i, 1, j] = np.std(x)
            features[i, 2, j] = np.ptp(x)
            features[i, 3, j] = kurtosis(
                x,
                fisher=False,
                bias=False
            )

    return features

In [53]:
train_features = extract_stat_features(X_train_audit)
val_features = extract_stat_features(X_val_audit)
test_features = extract_stat_features(X_test_audit)

print("Train:", train_features.shape)
print("Val:  ", val_features.shape)
print("Test: ", test_features.shape)

Train: (944, 4, 12)
Val:   (288, 4, 12)
Test:  (288, 4, 12)


In [54]:
# ============================================================
# Similarity Audit 1C: Train-based feature standardization
# Statistics: 4 Features x 12 Sensors
# ============================================================

feature_mean = train_features.mean(axis=0)
feature_std = train_features.std(axis=0)

feature_std[feature_std == 0] = 1.0

train_features_scaled = (train_features - feature_mean) / feature_std
val_features_scaled = (val_features - feature_mean) / feature_std
test_features_scaled = (test_features - feature_mean) / feature_std

print("Feature mean:", feature_mean.shape)
print("Feature std: ", feature_std.shape)

print("Train:", train_features_scaled.shape)
print("Val:  ", val_features_scaled.shape)
print("Test: ", test_features_scaled.shape)

Feature mean: (4, 12)
Feature std:  (4, 12)
Train: (944, 4, 12)
Val:   (288, 4, 12)
Test:  (288, 4, 12)


In [55]:
print("Maximum absolute Train mean:",
    np.abs(train_features_scaled.mean(axis=0)).max())

print("Minimum Train std:",
    train_features_scaled.std(axis=0).min())

print("Maximum Train std:",
    train_features_scaled.std(axis=0).max())

Maximum absolute Train mean: 4.543799423346708e-16
Minimum Train std: 0.9999999999999987
Maximum Train std: 1.0000000000000016


In [56]:
# ============================================================
# Similarity Audit 2: Train-Test distribution comparison
# All scenarios x features x sensors
# ============================================================

from scipy.stats import ks_2samp, wasserstein_distance

distribution_rows = []

for scenario in sorted(train_audit_meta["scenario"].unique()):

    train_mask = (train_audit_meta["scenario"].to_numpy() == scenario )
    test_mask = (test_audit_meta["scenario"].to_numpy() == scenario)
    case_name = train_audit_meta.loc[train_mask, "case"].iloc[0]

    for feature_idx, feature_name in enumerate(feature_names):

        for sensor_idx, sensor_name in enumerate(selected_sensor_names):

            train_values = train_features_scaled[train_mask, feature_idx, sensor_idx]
            test_values = test_features_scaled[test_mask, feature_idx, sensor_idx ]
            ks_stat, ks_p = ks_2samp(train_values,test_values )
            wasserstein = wasserstein_distance(train_values,test_values)

            distribution_rows.append({
                "case": case_name,
                "scenario": scenario,
                "feature": feature_name,
                "sensor": sensor_name,
                "ks_statistic": ks_stat,
                "ks_p_value": ks_p,
                "wasserstein": wasserstein
            })


distribution_df = pd.DataFrame(distribution_rows)

print("Number of comparisons:", len(distribution_df))

display(distribution_df.head(20))

Number of comparisons: 432


,case,scenario,feature,sensor,ks_statistic,ks_p_value,wasserstein
0,shm01a,1,RMS,DA04,0.234375,0.447371,0.051090
1,shm01a,1,RMS,DA05,0.234375,0.447371,0.068359
2,shm01a,1,RMS,DA06,0.171875,0.813784,0.052912
3,shm01a,1,RMS,DA07,0.375000,0.045544,0.181781
4,shm01a,1,RMS,DA08,0.218750,0.535416,0.061269
5,shm01a,1,RMS,DA09,0.234375,0.447371,0.118832
6,shm01a,1,RMS,DA10,0.296875,0.186702,0.003491
7,shm01a,1,RMS,DA11,0.343750,0.083551,0.407883
8,shm01a,1,RMS,DA12,0.328125,0.110759,0.622843
9,shm01a,1,RMS,DA13,0.203125,0.628647,0.083467


In [57]:
# ============================================================
# Similarity Audit 3: Statistical summary by scenario
# ============================================================

scenario_summary_df = (
    distribution_df
    .groupby(["case", "scenario"])
    .agg(
        mean_ks=("ks_statistic", "mean"),
        median_ks=("ks_statistic", "median"),
        mean_wasserstein=("wasserstein", "mean"),
        median_wasserstein=("wasserstein", "median"),
        significant_percent=(
            "ks_p_value",
            lambda x: 100 * np.mean(x < 0.05)
        )
    )
    .reset_index()
)

display(scenario_summary_df)

,case,scenario,mean_ks,median_ks,mean_wasserstein,median_wasserstein,significant_percent
0,shm01a,1,0.265299,0.250000,0.149694,0.090862,8.333333
1,shm02a,2,0.229167,0.218750,0.074357,0.064267,2.083333
2,shm03a,3,0.398112,0.390625,0.224741,0.237116,58.333333
3,shm04a,4,0.389648,0.398438,0.125174,0.114658,52.083333
4,shm05a,5,0.318685,0.312500,0.194522,0.118245,25.000000
5,shm06a,6,0.532986,0.552083,0.374355,0.361685,77.083333
6,shm07a,7,0.120443,0.117188,0.057827,0.049810,4.166667
7,shm08a,8,0.144531,0.135417,0.095098,0.090376,20.833333
8,shm09a,9,0.158746,0.166667,0.140281,0.130621,35.416667


In [58]:
# ============================================================
# Similarity Audit 4: Final statistical summary
# Block Random
# ============================================================

block_stat_summary = pd.DataFrame({
    "metric": [
        "Median KS statistic",
        "Mean KS statistic",
        "Median Wasserstein distance",
        "Mean Wasserstein distance"
    ],

    "value": [
        scenario_summary_df["median_ks"].median(),
        scenario_summary_df["mean_ks"].mean(),
        scenario_summary_df["median_wasserstein"].median(),
        scenario_summary_df["mean_wasserstein"].mean()
    ]
})

display(block_stat_summary)

,metric,value
0,Median KS statistic,0.250000
1,Mean KS statistic,0.284180
2,Median Wasserstein distance,0.114658
3,Mean Wasserstein distance,0.159561


In [59]:
# ============================================================
# Similarity Audit 5: Train-Test spectral similarity
# All scenarios x all sensors
# ============================================================

from scipy.signal import welch
from scipy.spatial.distance import cosine

spectral_rows = []

for scenario in sorted(train_audit_meta["scenario"].unique()):

    train_mask = (
        train_audit_meta["scenario"].to_numpy() == scenario
    )

    test_mask = (
        test_audit_meta["scenario"].to_numpy() == scenario
    )

    X_train_case = X_train_audit[train_mask]
    X_test_case = X_test_audit[test_mask]

    case_name = train_audit_meta.loc[
        train_mask, "case"
    ].iloc[0]

    fs = all_cases[case_name]["fs"]

    for sensor_idx, sensor_name in enumerate(selected_sensor_names):

        train_psd_list = []
        test_psd_list = []

        for window in X_train_case:

            f, psd = welch(
                window[:, sensor_idx],
                fs=fs,
                nperseg=256,
                noverlap=128
            )

            train_psd_list.append(psd)

        for window in X_test_case:

            f, psd = welch(
                window[:, sensor_idx],
                fs=fs,
                nperseg=256,
                noverlap=128
            )

            test_psd_list.append(psd)

        train_psd_list = np.array(train_psd_list)
        test_psd_list = np.array(test_psd_list)

        # Representative PSD of each split
        train_median_psd = np.median(
            train_psd_list,
            axis=0
        )

        test_median_psd = np.median(
            test_psd_list,
            axis=0
        )

        # PSD cosine similarity
        cosine_sim = 1 - cosine(
            train_median_psd,
            test_median_psd
        )

        # Log-PSD correlation
        eps = 1e-20

        train_log = np.log10(
            train_median_psd + eps
        )

        test_log = np.log10(
            test_median_psd + eps
        )

        log_corr = np.corrcoef(
            train_log,
            test_log
        )[0, 1]

        spectral_rows.append({
            "case": case_name,
            "scenario": scenario,
            "sensor": sensor_name,
            "psd_cosine_similarity": cosine_sim,
            "log_psd_correlation": log_corr
        })


spectral_df = pd.DataFrame(spectral_rows)

print(
    "Number of spectral comparisons:",
    len(spectral_df)
)

display(spectral_df.head(20))

Number of spectral comparisons: 108


,case,scenario,sensor,psd_cosine_similarity,log_psd_correlation
0,shm01a,1,DA04,0.999402,0.989838
1,shm01a,1,DA05,0.999980,0.991423
2,shm01a,1,DA06,0.999290,0.992819
3,shm01a,1,DA07,0.998133,0.992418
4,shm01a,1,DA08,0.999996,0.995201
5,shm01a,1,DA09,0.995312,0.993918
6,shm01a,1,DA10,0.989861,0.992088
7,shm01a,1,DA11,0.976691,0.990825
8,shm01a,1,DA12,0.985996,0.988695
9,shm01a,1,DA13,0.999580,0.993711


In [60]:
# ============================================================
# Similarity Audit 6: Spectral summary by scenario
# ============================================================

spectral_scenario_df = (
    spectral_df
    .groupby(["case", "scenario"])
    .agg(
        mean_psd_cosine=("psd_cosine_similarity", "mean"),
        median_psd_cosine=("psd_cosine_similarity", "median"),
        mean_log_psd_corr=("log_psd_correlation", "mean"),
        median_log_psd_corr=("log_psd_correlation", "median")
    )
    .reset_index()
)

display(spectral_scenario_df)

,case,scenario,mean_psd_cosine,median_psd_cosine,mean_log_psd_corr,median_log_psd_corr
0,shm01a,1,0.994553,0.998712,0.992523,0.992619
1,shm02a,2,0.990305,0.992353,0.990417,0.990659
2,shm03a,3,0.787277,0.835054,0.971725,0.973246
3,shm04a,4,0.890624,0.860788,0.983375,0.982740
4,shm05a,5,0.997107,0.999699,0.991642,0.991961
5,shm06a,6,0.980167,0.989595,0.994284,0.995009
6,shm07a,7,0.995423,0.996082,0.997928,0.998187
7,shm08a,8,0.983306,0.987801,0.996604,0.996938
8,shm09a,9,0.984192,0.998193,0.997663,0.998090


In [61]:
# ============================================================
# Similarity Audit 7: Final spectral summary
# Block Random
# ============================================================

block_spectral_summary = pd.DataFrame({
    "metric": [
        "Median PSD cosine similarity",
        "Mean PSD cosine similarity",
        "Median log-PSD correlation",
        "Mean log-PSD correlation"
    ],

    "value": [
        spectral_scenario_df["median_psd_cosine"].median(),
        spectral_scenario_df["mean_psd_cosine"].mean(),
        spectral_scenario_df["median_log_psd_corr"].median(),
        spectral_scenario_df["mean_log_psd_corr"].mean()
    ]
})

display(block_spectral_summary)

,metric,value
0,Median PSD cosine similarity,0.992353
1,Mean PSD cosine similarity,0.955884
2,Median log-PSD correlation,0.992619
3,Mean log-PSD correlation,0.990684


In [37]:
print("Train windows per case:")

display(
    train_meta_df
    .groupby("case")
    .size()
    .reset_index(name="windows")
)


print("Validation windows per case:")

display(
    val_meta_df
    .groupby("case")
    .size()
    .reset_index(name="windows")
)


print("Test windows per case:")

display(
    test_meta_df
    .groupby("case")
    .size()
    .reset_index(name="windows")
)

Train windows per case:


,case,windows
0,shm01a,64
1,shm02a,64
2,shm03a,64
3,shm04a,64
4,shm05a,64
5,shm06a,48
6,shm07a,192
7,shm08a,192
8,shm09a,192


Validation windows per case:


,case,windows
0,shm01a,16
1,shm02a,16
2,shm03a,16
3,shm04a,16
4,shm05a,16
5,shm06a,16
6,shm07a,64
7,shm08a,64
8,shm09a,64


Test windows per case:


,case,windows
0,shm01a,16
1,shm02a,16
2,shm03a,16
3,shm04a,16
4,shm05a,16
5,shm06a,16
6,shm07a,64
7,shm08a,64
8,shm09a,64


In [ ]:
import numpy as np
import torch

from torch.utils.data import TensorDataset, DataLoader


# ----------------------------
# بررسی داده‌ها قبل از Tensor
# ----------------------------
print("X_train shape:", X_train.shape)
print("X_val shape:  ", X_val.shape)
print("X_test shape: ", X_test.shape)

print("y_train shape:", y_train.shape)
print("y_val shape:  ", y_val.shape)
print("y_test shape: ", y_test.shape)


# تعداد داده و برچسب باید برابر باشد
assert len(X_train) == len(y_train)
assert len(X_val) == len(y_val)
assert len(X_test) == len(y_test)

# بررسی NaN و Inf
assert np.isfinite(X_train).all(), "X_train contains NaN or Inf."
assert np.isfinite(X_val).all(), "X_val contains NaN or Inf."
assert np.isfinite(X_test).all(), "X_test contains NaN or Inf."

print("Data check completed.")

X_train shape: (944, 512, 12)
X_val shape:   (288, 512, 12)
X_test shape:  (288, 512, 12)
y_train shape: (944,)
y_val shape:   (288,)
y_test shape:  (288,)
Data check completed.


In [ ]:
# ============================================================
# Step 6: Convert arrays to PyTorch tensors
# ============================================================

import random
import numpy as np
import torch

from torch.utils.data import TensorDataset, DataLoader


set_seed(MODEL_SEED)

# Inputs must be float32
X_train_tensor = torch.from_numpy(X_train.astype(np.float32, copy=False))
X_val_tensor = torch.from_numpy(X_val.astype(np.float32, copy=False))
X_test_tensor = torch.from_numpy(X_test.astype(np.float32, copy=False))


# Labels for CrossEntropyLoss must be int64
y_train_tensor = torch.from_numpy(y_train.astype(np.int64, copy=False))
y_val_tensor = torch.from_numpy(y_val.astype(np.int64, copy=False))
y_test_tensor = torch.from_numpy(y_test.astype(np.int64, copy=False))


print("\nTensor shapes:")
print("X_train_tensor:", X_train_tensor.shape)
print("y_train_tensor:", y_train_tensor.shape)

print("X_val_tensor:", X_val_tensor.shape)
print("y_val_tensor:", y_val_tensor.shape)

print("X_test_tensor:", X_test_tensor.shape)
print("y_test_tensor:", y_test_tensor.shape)


Tensor shapes:
X_train_tensor: torch.Size([944, 512, 12])
y_train_tensor: torch.Size([944])
X_val_tensor: torch.Size([288, 512, 12])
y_val_tensor: torch.Size([288])
X_test_tensor: torch.Size([288, 512, 12])
y_test_tensor: torch.Size([288])


In [ ]:
# ============================================================
# Create Dataset and DataLoader
# ============================================================

train_dataset = TensorDataset(X_train_tensor,y_train_tensor)
val_dataset   = TensorDataset(X_val_tensor,y_val_tensor)
test_dataset  = TensorDataset(X_test_tensor,y_test_tensor)

BATCH_SIZE = 64

# برای تکرارپذیری Shuffle داده‌های Train
loader_generator = torch.Generator()
loader_generator.manual_seed(MODEL_SEED)


train_loader = DataLoader(train_dataset,batch_size=BATCH_SIZE,shuffle=True,generator=loader_generator)
val_loader   = DataLoader(val_dataset,batch_size=BATCH_SIZE,shuffle=False)
test_loader  = DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False)

In [ ]:
print("Number of train samples:", len(train_dataset))
print("Number of val samples:  ", len(val_dataset))
print("Number of test samples: ", len(test_dataset))

print("\nNumber of train batches:", len(train_loader))
print("Number of val batches:  ", len(val_loader))
print("Number of test batches: ", len(test_loader))


X_batch, y_batch = next(iter(train_loader))

print("\nOne train batch:")
print("X_batch shape:", X_batch.shape)
print("y_batch shape:", y_batch.shape)
print("First 10 labels:", y_batch[:10])

Number of train samples: 944
Number of val samples:   288
Number of test samples:  288

Number of train batches: 15
Number of val batches:   5
Number of test batches:  5

One train batch:
X_batch shape: torch.Size([64, 512, 12])
y_batch shape: torch.Size([64])
First 10 labels: tensor([6, 6, 7, 2, 7, 7, 6, 5, 8, 7])


In [ ]:
# ============================================================
# Step 7: Define raw time-series Transformer
# Input shape: (batch, time, sensors)
# ============================================================

import torch.nn as nn
from sklearn.metrics import accuracy_score, f1_score

# ----------------------------
# Device
# ----------------------------
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)


# ----------------------------
# Learnable positional encoding
# ----------------------------
class LearnablePositionalEncoding(nn.Module):

    def __init__(self, seq_len, d_model):
        super().__init__()
        self.position = nn.Parameter(torch.zeros(1, seq_len, d_model))
        nn.init.trunc_normal_(self.position,std=0.02)

    def forward(self, x):
        seq_len = x.shape[1]
        return x + self.position[:, :seq_len, :]

Device: cpu


In [ ]:
# ============================================================
# Step 7: Define raw time-series Transformer
# Input shape: (batch, time, sensors)
# ============================================================

import torch.nn as nn

# --- Added these definitions to ensure the model parameters are defined ---
NUM_CLASSES = 9
SEQ_LEN = WINDOW_SIZE
INPUT_DIM = X_train_tensor.shape[2]
D_MODEL = 64
NUM_HEADS = 8
NUM_LAYERS = 2
DIM_FEEDFORWARD = 64
DROPOUT = 0.10
# --- End of added definitions ---

class TransformerTimeSeriesClassifier(nn.Module):
    def __init__(
        self,
        input_dim,
        seq_len,
        num_classes,
        d_model=64,
        num_heads=8,
        num_layers=2,
        dim_feedforward=64,
        dropout=0.10
    ):
        super().__init__()

        self.seq_len = seq_len
        self.d_model = d_model

        self.input_projection = nn.Linear(input_dim, d_model)
        self.pos_embedding = nn.Parameter(torch.zeros(1, seq_len, d_model))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=num_heads,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation="relu",
            batch_first=True,
            norm_first=True
        )

        self.transformer_encoder = nn.TransformerEncoder(encoder_layer,num_layers=num_layers)
        self.final_norm = nn.LayerNorm(d_model)
        self.classifier = nn.Linear(d_model,num_classes)
        self._init_weights()

    def _init_weights(self):
        nn.init.normal_(self.pos_embedding, mean=0.0, std=0.02)

    def forward(self, x):
        # x shape: (batch, time, channels)

        x = self.input_projection(x)
        x = x + self.pos_embedding[:, :x.size(1), :]
        x = self.transformer_encoder(x)
        x = self.final_norm(x)
        # Mean pooling over time
        x = x.mean(dim=1)
        logits = self.classifier(x)
        return logits
set_seed(MODEL_SEED)
model = TransformerTimeSeriesClassifier(
    input_dim=INPUT_DIM,
    seq_len=SEQ_LEN,
    num_classes=NUM_CLASSES,
    d_model=D_MODEL,
    num_heads=NUM_HEADS,
    num_layers=NUM_LAYERS,
    dim_feedforward=DIM_FEEDFORWARD,
    dropout=DROPOUT
).to(device)

print(model)

# ----------------------------
# Count trainable parameters
# ----------------------------
def count_trainable_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print("\nTrainable parameters:", count_trainable_parameters(model))


# ----------------------------
# Check one forward pass
# ----------------------------
for xb, yb in train_loader:
    xb = xb.to(device)

    logits = model(xb)

    print("\nInput batch shape:", xb.shape)
    print("Output logits shape:", logits.shape)

    break

TransformerTimeSeriesClassifier(
  (input_projection): Linear(in_features=12, out_features=64, bias=True)
  (transformer_encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
        )
        (linear1): Linear(in_features=64, out_features=64, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=64, out_features=64, bias=True)
        (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (final_norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
  (classifier): Linear(in_features=64, out_features=9, bias=True)
)

Trainable parameters: 84745


/tmp/ipykernel_1633/2973285209.py:49: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer_encoder = nn.TransformerEncoder(encoder_layer,num_layers=num_layers)



Input batch shape: torch.Size([64, 512, 12])
Output logits shape: torch.Size([64, 9])


In [ ]:
# ============================================================
# Step 8: Loss function and optimizer
# ============================================================

class_counts = torch.bincount(y_train_tensor, minlength=NUM_CLASSES).float()
if torch.any(class_counts == 0):
    raise ValueError("At least one class has no training samples.")

class_weights = (len(y_train_tensor) / (NUM_CLASSES * class_counts))
class_weights = class_weights.to(device)

print("Class counts and weights:")

for class_id in range(NUM_CLASSES):

    print(
        f"Scenario {class_id + 1} | "
        f"count: {int(class_counts[class_id])} | "
        f"weight: {class_weights[class_id].item():.4f}"
    )

Class counts and weights:
Scenario 1 | count: 64 | weight: 1.6389
Scenario 2 | count: 64 | weight: 1.6389
Scenario 3 | count: 64 | weight: 1.6389
Scenario 4 | count: 64 | weight: 1.6389
Scenario 5 | count: 64 | weight: 1.6389
Scenario 6 | count: 48 | weight: 2.1852
Scenario 7 | count: 192 | weight: 0.5463
Scenario 8 | count: 192 | weight: 0.5463
Scenario 9 | count: 192 | weight: 0.5463


In [ ]:
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 1e-4
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(model.parameters(),lr=LEARNING_RATE,weight_decay=WEIGHT_DECAY)

print("Loss function:", criterion)
print("Optimizer:", optimizer.__class__.__name__)
print("Learning rate:", LEARNING_RATE)

Loss function: CrossEntropyLoss()
Optimizer: AdamW
Learning rate: 0.0001


In [ ]:
# ============================================================
# Step 9: Train Transformer
# Best model selection based on Validation Macro-F1
# ============================================================

from sklearn.metrics import accuracy_score, f1_score
import copy
import torch

NUM_EPOCHS = 100
PATIENCE = 20
GRAD_CLIP = 1.0
MIN_DELTA = 1e-4

LABELS_ORDER = np.arange(NUM_CLASSES)

transformer_history = {
    "epoch": [],
    "train_loss": [],
    "train_acc": [],
    "train_macro_f1": [],
    "val_loss": [],
    "val_acc": [],
    "val_macro_f1": [],
}

best_val_macro_f1 = -1.0
best_epoch = 0
best_model_state = None
epochs_without_improvement = 0


def run_one_epoch_transformer(model, data_loader, criterion, optimizer=None, device="cpu"):
    """
    If optimizer is given: training mode
    If optimizer is None: evaluation mode
    """

    is_training = optimizer is not None

    if is_training:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    all_preds = []
    all_labels = []

    for xb, yb in data_loader:
        xb = xb.to(device)
        yb = yb.to(device)

        if is_training:
            optimizer.zero_grad()

        with torch.set_grad_enabled(is_training):
            logits = model(xb)
            loss = criterion(logits, yb)

            if is_training:
                loss.backward()

                # Gradient clipping helps Transformer training stability
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP)

                optimizer.step()

        total_loss += loss.item() * xb.size(0)
        preds = torch.argmax(logits, dim=1)
        all_preds.append(preds.detach().cpu())
        all_labels.append(yb.detach().cpu())

    avg_loss = total_loss / len(data_loader.dataset)
    all_preds = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()
    acc = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(
        all_labels,
        all_preds,
        average="macro",
        zero_division=0
    )

    return avg_loss, acc, macro_f1


for epoch in range(1, NUM_EPOCHS + 1):

    train_loss, train_acc, train_macro_f1 = run_one_epoch_transformer(
        model=model,
        data_loader=train_loader,
        criterion=criterion,
        optimizer=optimizer,
        device=device
    )

    val_loss, val_acc, val_macro_f1 = run_one_epoch_transformer(
        model=model,
        data_loader=val_loader,
        criterion=criterion,
        optimizer=None,
        device=device
    )

    transformer_history["epoch"].append(epoch)
    transformer_history["train_loss"].append(train_loss)
    transformer_history["train_acc"].append(train_acc)
    transformer_history["train_macro_f1"].append(train_macro_f1)
    transformer_history["val_loss"].append(val_loss)
    transformer_history["val_acc"].append(val_acc)
    transformer_history["val_macro_f1"].append(val_macro_f1)

    # Save best model based on validation Macro-F1
    if val_macro_f1 > best_val_macro_f1 + MIN_DELTA:
        best_val_macro_f1 = val_macro_f1
        best_epoch = epoch
        best_model_state = copy.deepcopy(model.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epoch == 1 or epoch % 5 == 0:
        print(
            f"Epoch {epoch:03d} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Train Acc: {train_acc:.4f} | "
            f"Train Macro-F1: {train_macro_f1:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val Acc: {val_acc:.4f} | "
            f"Val Macro-F1: {val_macro_f1:.4f}"
        )

    # Early stopping
    if epochs_without_improvement >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch}.")
        break



Epoch 001 | Train Loss: 2.2004 | Train Acc: 0.2786 | Train Macro-F1: 0.0984 | Val Loss: 2.0269 | Val Acc: 0.4306 | Val Macro-F1: 0.1705
Epoch 005 | Train Loss: 1.7818 | Train Acc: 0.4216 | Train Macro-F1: 0.2920 | Val Loss: 1.6359 | Val Acc: 0.4757 | Val Macro-F1: 0.3515
Epoch 010 | Train Loss: 1.1427 | Train Acc: 0.7606 | Train Macro-F1: 0.7081 | Val Loss: 1.0111 | Val Acc: 0.7708 | Val Macro-F1: 0.7198
Epoch 015 | Train Loss: 0.7312 | Train Acc: 0.9311 | Train Macro-F1: 0.9546 | Val Loss: 0.6716 | Val Acc: 0.9097 | Val Macro-F1: 0.9033


In [ ]:
if best_model_state is None:
    raise RuntimeError("No best model was saved.")

model.load_state_dict(best_model_state)

print("\nTraining finished.")
print("Best epoch:",best_epoch)
print("Best validation Macro-F1:",best_val_macro_f1)

In [ ]:
# ============================================================
# Final evaluation control
# ============================================================

# Keep False during development/tuning.
# Set True only for final test evaluation.
RUN_TEST = False

In [ ]:
# ============================================================
# Final evaluation
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report
)

import pandas as pd
import torch


train_eval_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)


def predict_with_model_transformer(
    model,
    data_loader,
    device="cpu"
):
    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():

        for xb, yb in data_loader:

            xb = xb.to(device)
            yb = yb.to(device)

            logits = model(xb)
            preds = torch.argmax(logits,dim=1)
            all_preds.append(preds.cpu())
            all_labels.append(yb.cpu())

    all_preds = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()

    return all_labels, all_preds


def final_evaluate_transformer(
    model,
    data_loader,
    split_name,
    device="cpu"
):

    y_true, y_pred = predict_with_model_transformer(
        model=model,
        data_loader=data_loader,
        device=device
    )

    acc = accuracy_score(
        y_true,
        y_pred
    )

    macro_f1 = f1_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

    weighted_f1 = f1_score(
        y_true,
        y_pred,
        average="weighted",
        zero_division=0
    )

    print(f"\n===== {split_name} =====")
    print(f"Accuracy:    {acc:.4f}")
    print(f"Macro-F1:    {macro_f1:.4f}")
    print(f"Weighted-F1: {weighted_f1:.4f}")

    return {
        "split": split_name,
        "accuracy": acc,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
        "y_true": y_true,
        "y_pred": y_pred
    }

In [ ]:
model.eval()

transformer_train_results = final_evaluate_transformer(
    model,
    train_eval_loader,
    "TRAIN",
    device=device
)

transformer_val_results = final_evaluate_transformer(
    model,
    val_loader,
    "VALIDATION",
    device=device
)

In [ ]:
transformer_test_results = None

if RUN_TEST:

    transformer_test_results = final_evaluate_transformer(
        model,
        test_loader,
        "TEST",
        device=device
    )

else:

    print("\nTest evaluation skipped.")

In [ ]:
summary_rows = [
    {
        "model": "Raw Transformer",
        "preprocessing":
            "Butterworth high-pass + Train-based normalization",
        "selected_sensors":
            len(selected_sensor_names),
        "split":
            transformer_train_results["split"],
        "accuracy":
            transformer_train_results["accuracy"],
        "macro_f1":
            transformer_train_results["macro_f1"],
        "weighted_f1":
            transformer_train_results["weighted_f1"]
    },

    {
        "model": "Raw Transformer",
        "preprocessing":
            "Butterworth high-pass + Train-based normalization",
        "selected_sensors":
            len(selected_sensor_names),
        "split":
            transformer_val_results["split"],
        "accuracy":
            transformer_val_results["accuracy"],
        "macro_f1":
            transformer_val_results["macro_f1"],
        "weighted_f1":
            transformer_val_results["weighted_f1"]
    }
]

In [ ]:
if RUN_TEST:

    summary_rows.append(
        {
            "model": "Raw Transformer",
            "preprocessing":
                "Butterworth high-pass + Train-based normalization",
            "selected_sensors":
                len(selected_sensor_names),
            "split":
                transformer_test_results["split"],
            "accuracy":
                transformer_test_results["accuracy"],
            "macro_f1":
                transformer_test_results["macro_f1"],
            "weighted_f1":
                transformer_test_results["weighted_f1"]
        }
    )

In [ ]:
transformer_summary_results = pd.DataFrame(
    summary_rows
)

display(
    transformer_summary_results
)

In [ ]:
# ============================================================
# Step 11: Test confusion matrix
# ============================================================


if RUN_TEST:

    y_true_test = (
        transformer_test_results["y_true"]
    )

    y_pred_test = (
        transformer_test_results["y_pred"]
    )

    cm_transformer_test = confusion_matrix(
        y_true_test,
        y_pred_test,
        labels=np.arange(NUM_CLASSES)
    )

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm_transformer_test,
        display_labels=[
            f"S{i+1}"
            for i in range(NUM_CLASSES)
        ]
    )

    disp.plot(
        values_format="d",
        cmap="Blues",
        colorbar=False
    )

    plt.title(
        "Test Confusion Matrix"
    )

    plt.grid(False)
    plt.show()

In [ ]:
# ============================================================
# Step 12: Training curves
# ============================================================

plt.figure(figsize=(8, 5))

plt.plot(
    transformer_history["epoch"],
    transformer_history["train_loss"],
    label="Train Loss"
)

plt.plot(
    transformer_history["epoch"],
    transformer_history["val_loss"],
    label="Validation Loss"
)

plt.axvline(
    best_epoch,
    linestyle="--",
    label=f"Best epoch = {best_epoch}"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    transformer_history["epoch"],
    transformer_history["train_macro_f1"],
    label="Train Macro-F1"
)

plt.plot(
    transformer_history["epoch"],
    transformer_history["val_macro_f1"],
    label="Validation Macro-F1"
)

plt.axvline(
    best_epoch,
    linestyle="--",
    label=f"Best epoch = {best_epoch}"
)

plt.xlabel("Epoch")
plt.ylabel("Macro-F1")
plt.title("Training and Validation Macro-F1")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()